In [5]:
# Import required libraries
import numpy as np
import scipy
import scipy.linalg as sla
import sparseqr
import matplotlib.pyplot as plt

In [6]:
# Read the required file | Roll No: 23b2157
roll_no = "23b2157.npz"
with np.load(roll_no) as system:
    u = system["x"]
    f = system["b"]
    K = scipy.sparse.coo_matrix(
        (system["A_values"], list(system["A_indices"])),
        shape=(u.size, u.size)
    )

In [7]:
# Check K * u = f
if(np.allclose(K.dot(u), f)):
    print("Everything is correct!")
else:
    print("There is something wrong!")

print("K shape:",K.shape)
print("u shape:",u.shape)
print("f shape:",f.shape)

Everything is correct!
K shape: (1494, 1494)
u shape: (1494,)
f shape: (1494,)


In [8]:
# QR factorisation for sparse matrix using sparseqr
# Returns Q, R, E (permutation vector), and rank
Q, R, E, rank = sparseqr.qr(K)

In [9]:
# Estimating explicit FLOPs for sparse QR is complex due to Householder reflections.
# We report the non-zeros of R as a proxy for structural complexity.
print("Non-zeros in R:", R.nnz)

Non-zeros in R: 267491


In [10]:
# Record the factorisation time using timeit
qr_factor_time_sparse = %timeit -o sparseqr.qr(K)

31.7 ms ± 3.43 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [11]:
print("Average Time:", qr_factor_time_sparse.average)
print("Minimum Time (No noise):", qr_factor_time_sparse.best)

Average Time: 0.03165764701428446
Minimum Time (No noise): 0.027630712500001663


In [12]:
# Solve the system using the internal SuiteSparse C-library solver
u_solved_sparse = sparseqr.solve(K, f, tolerance=0)
residual_sparse = np.linalg.norm(K.dot(u_solved_sparse) - f)

In [13]:
print("Residual:", residual_sparse)

Residual: 1.678651994840113e-08


In [14]:
# Record the time required to solve
qr_solve_time_sparse = %timeit -o sparseqr.solve(K, f, tolerance=0)

12.8 ms ± 311 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [15]:
print("Average Time:", qr_solve_time_sparse.average)
print("Minimum Time:", qr_solve_time_sparse.best)

Average Time: 0.012825086962857151
Minimum Time: 0.012476515829999925


In [16]:
K_dense = K.toarray()

qr_dense_factor_time = %timeit -o sla.qr(K_dense)

196 ms ± 39 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [17]:
# Dense FLOPs — theoretical formula for square QR
n = K_dense.shape[0]
flops_factor_dense = (4/3) * n**3
print("Factorization FlOps:", flops_factor_dense)

Factorization FlOps: 4446215712.0


In [18]:
Q_dense, R_dense = sla.qr(K_dense)
print("Average Time:", qr_dense_factor_time.average)
print("Minimum Time (No noise):", qr_dense_factor_time.best)

Average Time: 0.1959402022857343
Minimum Time (No noise): 0.16404329199997392


In [19]:
# Solve using dense matrices (Q^T * f and back substitution)
y = Q_dense.T @ f
u_solved_dense = sla.solve_triangular(R_dense, y)
residual_dense = np.linalg.norm(K_dense @ u_solved_dense - f)
print("Residual:", residual_dense)

Residual: 1.255452764865348e-08


In [20]:
# Record the time required to solve using dense matrices
qr_dense_solve_time = %timeit -o sla.solve_triangular(R_dense, Q_dense.T @ f)

1.1 ms ± 78 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [21]:
print("Average Time:", qr_dense_solve_time.average)
print("Minimum Time:", qr_dense_solve_time.best)
# Solve FLOPs: 2n^2 (Q.T @ f) + n^2 (back substitution)
print("FlOps:", 3 * n**2) 
print("GFlOpS:", (3 * n**2) / (qr_dense_solve_time.average * 10**9))

Average Time: 0.001102221880999975
Minimum Time: 0.0009811162920000242
FlOps: 6696108
GFlOpS: 6.075099864579944


In [22]:
# Memory for sparse
def memory_bytes(mat):
    if mat.format == 'coo':
        return mat.data.nbytes + mat.row.nbytes + mat.col.nbytes
    elif mat.format in ('csc', 'csr'):
        return mat.data.nbytes + mat.indices.nbytes + mat.indptr.nbytes
    else:
        return mat.tocoo().data.nbytes * 3

print("Storage for Sparse:\nK (COO):", memory_bytes(K) / 1024, "KB")
print("Q (COO):", memory_bytes(Q) / 1024, "KB")
print("R (COO):", memory_bytes(R) / 1024, "KB")

# Memory — dense storage
mem_dense_bytes = K_dense.nbytes   
print("Dense storage (MB):", mem_dense_bytes / 1024**2)

Storage for Sparse:
K (COO): 250.96875 KB
Q (COO): 11447.78125 KB
R (COO): 4179.546875 KB
Dense storage (MB): 17.029083251953125
